# 05 — Monitoramento de drift, retreino e promoção

Este notebook compara a base de referência com dados de 2026.

Depois ele retreina a pipeline até `2026-05-31` e registra uma nova versão no MLflow.

In [ ]:
from pathlib import Path
import json
import os
from datetime import datetime
import warnings

import joblib
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient
from scipy.stats import ks_2samp
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

In [ ]:
def find_lesson_dir():
    current = Path.cwd()
    if current.name == "4_Ciclo_Vida_MLOps_Bitcoin":
        return current
    candidate = current / "4_Ciclo_Vida_MLOps_Bitcoin"
    if candidate.exists():
        return candidate
    return current


LESSON_DIR = find_lesson_dir()
api_root = LESSON_DIR / "02_api_bitcoin"
if str(api_root) not in os.sys.path:
    os.sys.path.insert(0, str(api_root))

RUNTIME_DIR = LESSON_DIR / "runtime"
DATA_DIR = RUNTIME_DIR / "data"
ARTIFACTS_DIR = RUNTIME_DIR / "artifacts"
DRIFT_DIR = ARTIFACTS_DIR / "drift"
MLFLOW_RUNS_DIR = RUNTIME_DIR / "mlruns"
for directory in [RUNTIME_DIR, DATA_DIR, ARTIFACTS_DIR, DRIFT_DIR, MLFLOW_RUNS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BTC_URL = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"
MODEL_NAME = "bitcoin_price_forecaster"
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", f"sqlite:///{RUNTIME_DIR / 'mlflow.db'}")
mlflow.set_tracking_uri(TRACKING_URI)

In [ ]:
EXTRA_NUMERIC_COLUMNS = [
    "AdrActCnt",
    "AdrBalCnt",
    "BlkCnt",
    "CapMrktCurUSD",
    "HashRate",
    "ROI30d",
    "ROI1yr",
    "SplyCur",
    "TxCnt",
    "TxTfrCnt",
    "volume_reported_spot_usd_1d",
]


def load_btc_data(start_date, end_date):
    raw_path = DATA_DIR / "btc.csv"
    if raw_path.exists():
        data = pd.read_csv(raw_path)
    else:
        data = pd.read_csv(BTC_URL)
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        data.to_csv(raw_path, index=False)
    data["time"] = pd.to_datetime(data["time"])
    data = data.sort_values("time").reset_index(drop=True)
    mask = (data["time"] >= pd.Timestamp(start_date)) & (data["time"] <= pd.Timestamp(end_date))
    return data.loc[mask].copy()


def build_feature_table(data):
    frame = data.copy()
    frame["target_price_usd_d1"] = frame["PriceUSD"].shift(-1)
    frame["price_usd_current"] = frame["PriceUSD"]

    for lag in [1, 2, 3, 7, 14, 30]:
        frame[f"price_usd_lag_{lag}"] = frame["PriceUSD"].shift(lag)

    frame["return_1d"] = frame["PriceUSD"].pct_change()
    frame["log_return_1d"] = np.log(frame["PriceUSD"]).diff()

    for window in [7, 14, 30]:
        frame[f"rolling_mean_{window}"] = frame["PriceUSD"].rolling(window).mean()
        frame[f"rolling_std_{window}"] = frame["PriceUSD"].rolling(window).std()
        frame[f"rolling_min_{window}"] = frame["PriceUSD"].rolling(window).min()
        frame[f"rolling_max_{window}"] = frame["PriceUSD"].rolling(window).max()
        frame[f"rolling_return_{window}"] = frame["PriceUSD"].pct_change(window)

    frame["day_of_week"] = frame["time"].dt.dayofweek
    frame["month"] = frame["time"].dt.month
    frame["quarter"] = frame["time"].dt.quarter
    frame["year"] = frame["time"].dt.year

    feature_columns = [
        "price_usd_current",
        "price_usd_lag_1",
        "price_usd_lag_2",
        "price_usd_lag_3",
        "price_usd_lag_7",
        "price_usd_lag_14",
        "price_usd_lag_30",
        "return_1d",
        "log_return_1d",
        "day_of_week",
        "month",
        "quarter",
        "year",
    ]

    for window in [7, 14, 30]:
        feature_columns.extend(
            [
                f"rolling_mean_{window}",
                f"rolling_std_{window}",
                f"rolling_min_{window}",
                f"rolling_max_{window}",
                f"rolling_return_{window}",
            ]
        )

    for column in EXTRA_NUMERIC_COLUMNS:
        if column in frame.columns and frame[column].notna().mean() >= 0.80:
            feature_columns.append(column)

    selected = frame[["time", "target_price_usd_d1"] + feature_columns].copy()
    selected = selected.dropna(subset=["target_price_usd_d1"] + feature_columns).reset_index(drop=True)
    return selected, feature_columns

## Dados de referência e monitoramento

A referência é o período usado até 2025.

O monitoramento usa `2026-01-01` até `2026-05-31`.

In [ ]:
reference_raw = load_btc_data("2018-01-01", "2025-12-31")
monitoring_raw = load_btc_data("2026-01-01", "2026-05-31")

reference, feature_columns = build_feature_table(reference_raw)
monitoring, monitoring_feature_columns = build_feature_table(monitoring_raw)
common_features = [column for column in feature_columns if column in monitoring_feature_columns]

reference_X = reference[common_features]
monitoring_X = monitoring[common_features]

print(reference_X.shape, monitoring_X.shape)
print(monitoring[["time", "target_price_usd_d1"]].head())

## PSI e teste KS

O PSI compara a distribuição da referência com a distribuição nova.

O teste KS é uma checagem complementar para variáveis numéricas.

In [ ]:
def calculate_psi(expected, actual, buckets=10):
    expected = pd.Series(expected).dropna().astype(float)
    actual = pd.Series(actual).dropna().astype(float)
    if expected.empty or actual.empty:
        return np.nan

    quantiles = np.linspace(0, 1, buckets + 1)
    breakpoints = np.unique(np.quantile(expected, quantiles))
    if len(breakpoints) < 3:
        return 0.0

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]
    expected_perc = np.maximum(expected_counts / max(expected_counts.sum(), 1), 0.0001)
    actual_perc = np.maximum(actual_counts / max(actual_counts.sum(), 1), 0.0001)
    return float(np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc)))


drift_rows = []
for column in common_features:
    psi = calculate_psi(reference_X[column], monitoring_X[column])
    ks_stat, ks_pvalue = ks_2samp(reference_X[column], monitoring_X[column])
    drift_rows.append(
        {
            "feature": column,
            "psi": psi,
            "ks_statistic": float(ks_stat),
            "ks_pvalue": float(ks_pvalue),
            "drift_level": "alto" if psi >= 0.25 else "moderado" if psi >= 0.10 else "baixo",
        }
    )

drift_report = pd.DataFrame(drift_rows).sort_values("psi", ascending=False)
drift_report.to_csv(DRIFT_DIR / "drift_report_2026.csv", index=False)
drift_report.head(10)

## Avaliar modelo atual nos dados novos

Primeiro tentamos carregar o modelo `prod`.

Se não houver MLflow disponível, usamos o artefato local salvo pelo notebook 01.

In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.maximum((np.abs(y_true) + np.abs(y_pred)) / 2, 1)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "smape": float(np.mean(np.abs(y_true - y_pred) / denominator)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def load_current_model():
    for alias in ["prod", "homol"]:
        try:
            model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{alias}")
            return model, f"models:/{MODEL_NAME}@{alias}"
        except Exception:
            pass

    local_model_path = ARTIFACTS_DIR / "local_model.joblib"
    if local_model_path.exists():
        return joblib.load(local_model_path), str(local_model_path)
    return None, "unavailable"


current_model, current_model_uri = load_current_model()
monitoring_metrics = {"model_uri": current_model_uri}

if current_model is not None and not monitoring.empty:
    predictions = current_model.predict(monitoring_X[common_features])
    monitoring_metrics.update(calculate_metrics(monitoring["target_price_usd_d1"], predictions))
else:
    print("Modelo atual indisponível. Execute o notebook 01 antes desta etapa.")

(DRIFT_DIR / "monitoring_metrics_2026.json").write_text(
    json.dumps(monitoring_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
monitoring_metrics

## Retreinar com dados até 2026-05-31

O novo modelo vai para `homol`.

A promoção para `prod` deve acontecer apenas depois da homologação.

In [ ]:
full_raw = load_btc_data("2018-01-01", "2026-05-31")
full_supervised, full_features = build_feature_table(full_raw)

train_cutoff = "2025-12-31"
retrain_train = full_supervised[full_supervised["time"] <= train_cutoff].copy()
retrain_valid = full_supervised[full_supervised["time"] > train_cutoff].copy()

X_retrain = retrain_train[full_features]
y_retrain = retrain_train["target_price_usd_d1"]
X_revalid = retrain_valid[full_features]
y_revalid = retrain_valid["target_price_usd_d1"]

retrained_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(n_estimators=140, random_state=42, n_jobs=1, min_samples_leaf=3)),
    ]
)
retrained_model.fit(X_retrain, y_retrain)
revalid_pred = retrained_model.predict(X_revalid)
retrain_metrics = calculate_metrics(y_revalid, revalid_pred)
retrain_metrics

In [ ]:
RETRAIN_EXPERIMENT_NAME = "aula_4_bitcoin_retraining"
client = MlflowClient()
experiment = client.get_experiment_by_name(RETRAIN_EXPERIMENT_NAME)
if experiment is None:
    client.create_experiment(
        RETRAIN_EXPERIMENT_NAME,
        artifact_location=MLFLOW_RUNS_DIR.resolve().as_uri(),
    )
mlflow.set_experiment(RETRAIN_EXPERIMENT_NAME)
feature_path = ARTIFACTS_DIR / "feature_columns.json"
feature_path.write_text(json.dumps({"feature_columns": full_features}, indent=2, ensure_ascii=False), encoding="utf-8")

model_version = None
with mlflow.start_run(run_name=f"retreino_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
    mlflow.log_param("target", "target_price_usd_d1")
    mlflow.log_param("training_end", "2026-05-31")
    mlflow.log_param("feature_count", len(full_features))
    for name, value in retrain_metrics.items():
        mlflow.log_metric(f"monitoring_validation_{name}", value)
    mlflow.log_artifact(str(feature_path))

    input_example = X_retrain.head(3)
    signature = infer_signature(input_example, retrained_model.predict(input_example))
    model_info = mlflow.sklearn.log_model(
        retrained_model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=input_example,
        metadata={"feature_columns": json.dumps(full_features)},
        await_registration_for=60,
    )
    print("Model URI:", model_info.model_uri)


joblib.dump(retrained_model, ARTIFACTS_DIR / "local_model.joblib")
current_sample = {"features": X_revalid.iloc[-1].astype(float).to_dict()}
(ARTIFACTS_DIR / "sample_payload.json").write_text(
    json.dumps(current_sample, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

monitoring_predictions = retrain_valid[["time", "target_price_usd_d1"]].copy()
monitoring_predictions["prediction"] = revalid_pred
monitoring_predictions.to_csv(ARTIFACTS_DIR / "predictions_monitoring.csv", index=False)

current_metadata = {
    "model_name": MODEL_NAME,
    "selected_model": "extra_trees_retrained",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "target": "target_price_usd_d1",
    "feature_columns_path": str(feature_path.relative_to(LESSON_DIR)),
    "local_model_path": str((ARTIFACTS_DIR / "local_model.joblib").relative_to(LESSON_DIR)),
    "current_predictions_path": str((ARTIFACTS_DIR / "predictions_monitoring.csv").relative_to(LESSON_DIR)),
    "stage": "homol",
}
(ARTIFACTS_DIR / "model_metadata.json").write_text(
    json.dumps(current_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

try:
    versions = client.search_model_versions(f"name = '{MODEL_NAME}'")
    latest = max(versions, key=lambda version: int(version.version))
    model_version = latest.version
    client.set_registered_model_alias(MODEL_NAME, "homol", model_version)
    print(f"Alias homol atualizado para versão {model_version}.")
except Exception as exc:
    print("Não foi possível atualizar alias homol.")
    print(exc)

promote_to_prod = os.getenv("PROMOTE_TO_PROD", "").lower() in {"1", "true", "yes"}
if promote_to_prod and model_version:
    client.set_registered_model_alias(MODEL_NAME, "prod", model_version)
    print(f"Alias prod atualizado para versão {model_version}.")
else:
    print("Promoção para prod não executada. Defina PROMOTE_TO_PROD=true depois da homologação.")